# ETL Pipeline — Transactions Fraud Dataset → BigQuery

Notebook pobiera dane z Kaggle i ładuje je do Google BigQuery.

**Przed uruchomieniem:**
1. Zmień `YOUR-PROJECT-ID` na swój projekt GCP
2. Upewnij się, że masz włączone **BigQuery API** w projekcie
3. Zainstaluj wymagane biblioteki (komórka poniżej)

In [ ]:
!pip install kagglehub pandas-gbq

In [ ]:
import kagglehub
import pandas as pd
import os
from google.colab import auth
from google.cloud import bigquery

#  KONFIGURACJA — zmień na swój projekt GCP
project_id = 'YOUR-PROJECT-ID'  # <- wpisz swój Project ID z Google Cloud Console
dataset_id = 'tpay_fraud_project'
table_id   = f"{project_id}.{dataset_id}.transactions"

# 1. EXTRACT — autoryzacja i pobranie danych z Kaggle
auth.authenticate_user()
path = kagglehub.dataset_download("computingvictor/transactions-fraud-datasets")
csv_path = os.path.join(path, 'transactions_data.csv')
df = pd.read_csv(csv_path)

print(f"Pobrano {len(df):,} rekordów | Kolumny: {list(df.columns)}")

In [ ]:
# 2. TRANSFORM — normalizacja nazw kolumn (usunięcie kropek i spacji)
df.columns = [c.replace('.', '_').replace(' ', '_') for c in df.columns]

print(f" Znormalizowane kolumny: {list(df.columns)}")

In [ ]:
# 3. LOAD — upload do BigQuery
client  = bigquery.Client(project=project_id)
dataset = bigquery.Dataset(f"{project_id}.{dataset_id}")
client.create_dataset(dataset, exists_ok=True)

try:
    df.to_gbq(
        table_id,
        project_id=project_id,
        if_exists='replace',      # 'append' aby dodawać, 'replace' aby nadpisać
        use_bqstorage_api=True    # BigQuery Storage Write API — szybszy upload
    )
    print(f"Sukces! Dane załadowane do: {table_id}")
except Exception as e:
    print(f" Błąd podczas uploadu: {e}")